In [22]:
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel

In [23]:
import os
from enum import Enum
import pandas as pd
from datetime import datetime

In [24]:
load_dotenv()

True

In [25]:
LLM_API_URL = os.environ["LLM_API_URL"]
LLM_API_TOKEN = os.environ["LLM_API_TOKEN"]
MODEL = "google/gemma-3-1b"

In [26]:
client = OpenAI(
    base_url=LLM_API_URL,
    api_key=LLM_API_TOKEN
)

In [7]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Hi!"}]
)

print(response.choices[0].message.content)

Hi there! How can I help you today? 😊 

Do you have a question, need some information, or just want to chat?


In [27]:
VOID        = 0
PLAYER      = 1
ENNEMY      = 2
GOLD        = 3

SYMBOLS = {VOID: "·", PLAYER: "👤", ENNEMY: "👹", GOLD: "💰"}

In [28]:
# Carte 1 : L'autoroute (pour tester si l'IA va tout droit sans se perdre)
map_facile = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 3],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3],
    [0, 0, 0, 0, 0, 0, 0],
])

# Carte 2 : L'obstacle (pour tester l'esquive de base)
map_obstacle = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 2, 0, 3],
    [0, 0, 0, 0, 2, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
])

# Carte 3 : Le piège (pour tester la mémoire et le chemin complexe)
map_piege = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 2, 2, 3],
    [0, 0, 0, 0, 0, 2, 0],
    [0, 0, 2, 2, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
])

# # Couche de contrat

In [29]:
class Direction(str, Enum):
    HAUT       = "HAUT"
    BAS        = "BAS"
    GAUCHE     = "GAUCHE"
    DROITE     = "DROITE"


class PlayerDecision(BaseModel):
    analyse: str
    direction: Direction

In [30]:
MOVES = {
    "HAUT":     (-1, 0),
    "BAS":      ( 1,  0),
    "GAUCHE":   ( 0,  -1),
    "DROITE":   ( 0,   1),
}

# # Moteur de perception

In [31]:
def localize(world_map, entity):
    positions = np.argwhere(world_map == entity)
    return positions

In [32]:
def compute_distances(entities_positions, reference_pos):
    if (len(entities_positions) == 0):
        return np.array([])
    
    v = entities_positions - reference_pos
    distances = np.linalg.norm(v, axis=1)
 
    return np.round(distances, 2)

In [33]:
def perception(world_map, assist_mode=True):
    player_position = localize(world_map, PLAYER)[0]
    golds_positions = localize(world_map, GOLD)
    ennemies_positions = localize(world_map, ENNEMY)

    mouvements_possibles = []
    for direction, (d_row, d_col) in MOVES.items():
        new_pos = (player_position[0] + d_row, player_position[1] + d_col)
        if allowed_move(world_map, new_pos): 
            mouvements_possibles.append(direction)

    directions_vers_or = []
    if len(golds_positions) > 0:
        golds_distances = compute_distances(golds_positions, player_position)
        nearest_gold_idx = np.argmin(golds_distances)
        delta = golds_positions[nearest_gold_idx] - player_position
        
        if delta[0] > 0: directions_vers_or.append("BAS")
        elif delta[0] < 0: directions_vers_or.append("HAUT")
        if delta[1] > 0: directions_vers_or.append("DROITE")
        elif delta[1] < 0: directions_vers_or.append("GAUCHE")

    directions_dangereuses = []
    if len(ennemies_positions) > 0:
        ennemies_distances = compute_distances(ennemies_positions, player_position)
        nearest_enemy_idx = np.argmin(ennemies_distances)
        delta_ennemi = ennemies_positions[nearest_enemy_idx] - player_position
        
        if delta_ennemi[0] == 1 and delta_ennemi[1] == 0: directions_dangereuses.append("BAS")
        elif delta_ennemi[0] == -1 and delta_ennemi[1] == 0: directions_dangereuses.append("HAUT")
        elif delta_ennemi[0] == 0 and delta_ennemi[1] == 1: directions_dangereuses.append("DROITE")
        elif delta_ennemi[0] == 0 and delta_ennemi[1] == -1: directions_dangereuses.append("GAUCHE")

    # ---- LE FILTRE BENCHMARK ----
    if assist_mode:
        choix_recommandes = [d for d in directions_vers_or if d in mouvements_possibles and d not in directions_dangereuses]
        if not choix_recommandes:
            choix_recommandes = [d for d in mouvements_possibles if d not in directions_dangereuses]
    else:
        # En mode non assisté, le LLM n'a que la direction théorique de l'or. À lui d'éviter les murs !
        choix_recommandes = directions_vers_or if directions_vers_or else ["HAUT", "BAS", "GAUCHE", "DROITE"]

    return {
        "choix_recommandes": choix_recommandes
    }

In [34]:
def show_map(world_map):
    for row in world_map:
        print("\t".join(SYMBOLS.get(cell, "?") for cell in row))
    print('-----------------------------------------------------')

# # Moteur de déplacement

In [35]:
def allowed_move(world_map: np.ndarray, pos):
    n_rows, n_cols = world_map.shape
    r, c = pos

    if r < 0 or c < 0 or r >= n_rows or c >= n_cols:
        return False
    
    return world_map[r, c] in [VOID, GOLD]

In [36]:
def move(world_map: np.ndarray, old_pos, new_pos):
    if not allowed_move(world_map, new_pos):
        return old_pos
    
    entity = world_map[old_pos[0], old_pos[1]]
    world_map[old_pos[0], old_pos[1]] = VOID
    world_map[new_pos[0], new_pos[1]] = entity

    return new_pos

# # Moteur de décision

In [37]:
def decide(player_perception) -> PlayerDecision | None:
    # On récupère les choix recommandés et l'historique
    choix_possibles = player_perception['choix_recommandes'].copy()
    historique = player_perception.get('move_history', [])

    # --- FILTRE ANTI-PING-PONG ---
    if len(historique) > 0:
        dernier_mouvement = historique[-1]
        opposes = {"HAUT": "BAS", "BAS": "HAUT", "GAUCHE": "DROITE", "DROITE": "GAUCHE"}
        mouvement_inverse = opposes.get(dernier_mouvement)

        # Si l'IA a un autre choix que de faire demi-tour, on lui interdit le retour en arrière
        if mouvement_inverse in choix_possibles and len(choix_possibles) > 1:
            choix_possibles.remove(mouvement_inverse)
    # -----------------------------

    prompt = f"""
    Tu es l'IA d'un personnage. 
    
    Le système a calculé pour toi les seules directions sûres et optimales :
    {choix_possibles}

    MISSION :
    - analyse : Confirme ton choix.
    - direction : Choisis EXACTEMENT l'une des directions de la liste.
    """

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=PlayerDecision,
        temperature=0.0
    )
    
    decision = response.choices[0].message.parsed
    if decision:
        print(f"\t 🧠 Décision du LLM : {decision.analyse}")

    return decision

# # Game loop (simulation)


In [38]:
def game_loop(world_map: np.ndarray, map_name: str, assist_mode: bool = True, max_turns = 20):
    world_map = world_map.copy()
    move_history = []
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    logs = []
    
    # --- NOUVEAU : On compte l'or initial ---
    initial_gold = len(localize(world_map, GOLD)) 
    
    print(f"🎮 Lancement de la carte : {map_name} (Run: {run_id})")
    
    for turn in range(max_turns):
        print(f"\n =================== [Turn {turn + 1}] ===================")
        show_map(world_map)

        player_positions = localize(world_map, PLAYER)
        if len(player_positions) == 0:
            print("💥 Game Over : Le joueur a disparu !")
            break
        player_pos = player_positions[0]
        
        golds_positions = localize(world_map, GOLD)
        current_gold = len(golds_positions)
        
        # --- NOUVEAU : On calcule l'or ramassé ---
        coins_collected = initial_gold - current_gold 
        
        if current_gold == 0:
            print("🏆 Victoire absolue ! Toutes les pièces ont été ramassées !")
            break
            
        golds_distances = compute_distances(golds_positions, player_pos)
        nearest_gold_dist = np.min(golds_distances)

        p = perception(world_map, assist_mode=assist_mode)
        p["move_history"] = move_history

        decision: PlayerDecision | None = decide(p)

        if decision is not None:
            current_log = {
                "run_id": run_id,
                "map_name": map_name,
                "model_name": MODEL,
                "assist_mode": assist_mode,
                "turn": turn + 1,
                "player_row": int(player_pos[0]),
                "player_col": int(player_pos[1]),
                "decision": decision.direction.value,
                "nearest_gold_distance": float(nearest_gold_dist),
                "is_useless_move": False,
                "initial_gold": initial_gold,       # Ajout au Parquet
                "coins_collected": coins_collected  # Ajout au Parquet
            }

            move_history.append(decision.direction.value)

            d_row, d_col = MOVES[decision.direction.value]
            new_pos = (player_pos[0] + d_row, player_pos[1] + d_col)
            actual_new_pos = move(world_map, player_pos, new_pos)
            
            if player_pos[0] == actual_new_pos[0] and player_pos[1] == actual_new_pos[1]:
                current_log["is_useless_move"] = True
            
            logs.append(current_log)

    if len(logs) > 0:
        df = pd.DataFrame(logs)
        filename = f"benchmark_ia/data/bronze_{map_name}_{run_id}.parquet"
        df.to_parquet(filename)
        print(f"\n💾 Fin de simulation. Données ingérées dans : {filename}")
        
    return logs

In [39]:
import time

print("🚀 Démarrage de la campagne de Benchmark...")

# On lance 5 parties AVEC l'aide de l'algorithme Python
for i in range(5):
    print(f"\n--- Simulation ASSISTÉE {i+1}/5 ---")
    game_loop(world_map=map_obstacle, map_name="map_obstacle", assist_mode=True, max_turns=15)
    time.sleep(1) # Petite pause pour éviter de spammer l'API

# On lance 5 parties SANS l'aide de l'algorithme Python
for i in range(5):
    print(f"\n--- Simulation SURCHARGÉE {i+1}/5 ---")
    game_loop(world_map=map_obstacle, map_name="map_obstacle", assist_mode=False, max_turns=15)
    time.sleep(1)

print("\n✅ Campagne terminée ! Les fichiers Parquet sont prêts pour dbt.")

🚀 Démarrage de la campagne de Benchmark...

--- Simulation ASSISTÉE 1/5 ---
🎮 Lancement de la carte : map_obstacle (Run: 20260706_140603)

 =================== [Turn 1] ===================
·	·	·	·	·	·	·
·	👤	·	·	👹	·	💰
·	·	·	·	👹	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
-----------------------------------------------------
	 🧠 Décision du LLM : Je confirme mon choix.

 =================== [Turn 2] ===================
·	·	·	·	·	·	·
·	·	👤	·	👹	·	💰
·	·	·	·	👹	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
-----------------------------------------------------
	 🧠 Décision du LLM : Je suis prêt à analyser.

 =================== [Turn 3] ===================
·	·	·	·	·	·	·
·	·	·	👤	👹	·	💰
·	·	·	·	👹	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
-----------------------------------------------------
	 🧠 Décision du LLM : Je suis prêt à analyser.

 =================== [Turn 4] ===================
·	·	·	👤	·	·	·
·	·	·	·	👹	·	💰
·	·	·	·	👹	·	·
·	·	·	·	·	·	·
·	·	·	·